In [6]:
import pickle
import numpy as np
import pandas as pd
import os
from iblatlas.atlas import BrainRegions
from datetime import datetime
from one.api import ONE

one = ONE()

In [7]:
prefix = '/home/ines/repositories/'
# prefix = '/Users/ineslaranjeira/Documents/Repositories/'

# Load data

In [8]:
import brainwidemap
bwm_df = brainwidemap.bwm_query()
n_sessions = bwm_df['eid'].unique().shape[0]
n_insertions = bwm_df['pid'].unique().shape[0]
print(f'{n_sessions} sessions with {n_insertions} individual neuropixel recordings')
bwm_pid = bwm_df['pid'].unique()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
459 sessions with 699 individual neuropixel recordings


## LDA axis

In [9]:
data_path = prefix + 'representation_learning_variability/paper-individuality/clustering/'
lda = pd.read_pickle(data_path + 'mouse_LDA_5_bins_cut19-06-2026')
lda = lda.rename(columns={0: 'lda_1'})

lda_eid = lda.loc[lda['session'].isin(list(bwm_df.eid)), 'session']
lda_pid = bwm_df.loc[bwm_df['eid'].isin(lda_eid), 'pid']

print(f'Matching sessions: {len(lda_eid)}')
print(f'Matching insertions: {len(lda_pid)}')

Matching sessions: 244
Matching insertions: 380


In [10]:
# Initialize IBL Brain Atlas
regions = BrainRegions()

def get_simplified_area(col_name):
    """Traces an Allen ID/acronym back up to its Beryl-level parent region."""
    raw_acronym = col_name.split('_neuron_')[0]
    allen_ids = regions.acronym2id(raw_acronym)
    beryl_ids = regions.remap(allen_ids, source_map='Allen', target_map='Beryl')
    return regions.id2acronym(beryl_ids)[0]

## Extract firing rates per trial

In [11]:
# PSTH parameters based on 60 Hz sampling rate
MAX_TRIALS_TO_PROCESS = 400 
PSTH_PRE = 0.5   # 500 ms before event
PSTH_POST = 1.0  # 1000 ms after event
MIN_BINS_RATIO = 0.9  # Accept trials with ≥75% of expected bins

path_dir = '/Users/ineslaranjeira/Library/CloudStorage/GoogleDrive-ines.laranjeira@research.fchampalimaud.org/O meu disco/CCU/PhD project/paper-individuality/data/neuron_files/'
path_dir = prefix + 'representation_learning_variability/paper-individuality/data/neuron_files/'
save_dir = prefix + 'representation_learning_variability/paper-individuality/data/firing_rates/'

# Create save directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# --- Ground-truth stimulus side/contrast, fetched once per session and cached ---
# contrastLeft/contrastRight are mutually exclusive per trial (exactly one is non-NaN),
# including at 0% contrast, so this has no ambiguous cases unlike inferring side from
# choice+correct (which cannot distinguish sides at 0% contrast and, as it turns out,
# was also miscomputing side on every incorrect trial - see notebook discussion).
one_trials_cache = {}
failed_sessions = {}

def get_true_side_contrast(session_eid):
    """Returns (true_side, true_contrast) arrays indexed by trial_id for a session, or None on failure."""
    if session_eid in one_trials_cache:
        return one_trials_cache[session_eid]
    if session_eid in failed_sessions:
        return None
    try:
        trials = one.load_object(session_eid, 'trials', collection='alf')
        cl = np.asarray(trials['contrastLeft'], dtype=float)
        cr = np.asarray(trials['contrastRight'], dtype=float)
    except Exception as e:
        print(f"  Could not fetch trials for session {session_eid}: {e}")
        failed_sessions[session_eid] = str(e)
        return None

    is_left = ~np.isnan(cl)
    is_right = ~np.isnan(cr)
    # Sanity check: exactly one of contrastLeft/contrastRight should be set per trial.
    # Any trial that violates this (both set, or neither) has an undefined stimulus
    # side, so it is flagged as unresolved (None) rather than silently guessed.
    well_defined = is_left != is_right
    n_bad = int((~well_defined).sum())
    if n_bad:
        print(f"  Session {session_eid}: {n_bad}/{len(cl)} trials have ambiguous/missing "
              f"contrastLeft+contrastRight; those trials will be dropped, not guessed.")

    true_side = np.full(len(cl), None, dtype=object)
    true_side[well_defined & is_left] = 'Left'
    true_side[well_defined & is_right] = 'Right'
    true_contrast = np.where(is_left, cl, cr)

    result = (true_side, true_contrast)
    one_trials_cache[session_eid] = result
    return result


def align_bins(window_bin_times, window_spike_counts, event, pre, expected_bins, sampling_rate=60):
    """Aligns irregular bin timestamps onto the fixed reference time axis.

    Uses np.add.at to accumulate (and average) any bins that round to the same
    target position instead of silently overwriting on collision, and reports
    whether a collision happened so it can be logged and inspected rather than
    passing unnoticed.
    """
    sums = np.zeros(expected_bins)
    counts = np.zeros(expected_bins, dtype=int)

    bin_positions = np.round((window_bin_times - event + pre) * sampling_rate).astype(int)
    in_range = (bin_positions >= 0) & (bin_positions < expected_bins)

    bp = bin_positions[in_range]
    sc = window_spike_counts[in_range] * sampling_rate
    np.add.at(sums, bp, sc)
    np.add.at(counts, bp, 1)

    collided = bool(np.any(counts > 1))
    filled = counts > 0
    firing_rates_aligned = np.full(expected_bins, np.nan, dtype=float)
    firing_rates_aligned[filled] = sums[filled] / counts[filled]
    return firing_rates_aligned, int(filled.sum()), collided


processed_count = 0
skipped_count = 0

for pid in lda_pid:
    # Check if output file already exists
    save_filename = save_dir + "firing_rate_" + str(pid)
    if os.path.exists(save_filename):
        skipped_count += 1
        print(f"Skipping PID (already processed): {pid}")
        continue
    
    filepath = os.path.join(path_dir, f'states_neurons_file_{pid}')
    try:
        summary_records = []
        with open(filepath, 'rb') as f:
            raw_data = pickle.load(f)
            
        state_with_spikes = raw_data.dropna(subset=['Bin']).reset_index(drop=True)
        if state_with_spikes.empty:
            continue
            
        pid_name = state_with_spikes['session_pid'].iloc[0]
        session_name = state_with_spikes['session'].iloc[0]

        side_contrast = get_true_side_contrast(session_name)
        if side_contrast is None:
            print(f"  Skipping PID {pid_name}: no usable trials info for session {session_name}")
            continue
        true_side, true_contrast = side_contrast
        n_one_trials = len(true_side)

        # --- Resolve one row of (trial_id, event_time, condition) per trial, using
        # trial_id (an integer, unique-by-construction session-trial index) as the
        # join key instead of matching on raw float goCueTrigger_times. This removes
        # any chance of two trials silently colliding on a float dict key. ---
        trial_order = (state_with_spikes.drop_duplicates(subset=['trial_id'])
                       .sort_values('trial_id')[['trial_id', 'goCueTrigger_times']])
        trial_order['trial_id'] = trial_order['trial_id'].astype(int)
        assert trial_order['trial_id'].is_unique, f"Duplicate trial_id values in {filepath}"
        trial_order = trial_order.iloc[:MAX_TRIALS_TO_PROCESS]

        valid_trial_rows = []
        for tid, event in trial_order.itertuples(index=False):
            if not (0 <= tid < n_one_trials) or true_side[tid] is None:
                continue  # stimulus side undefined for this trial - drop it, don't guess
            condition = f"{true_side[tid]}_{true_contrast[tid]}"
            valid_trial_rows.append((tid, event, condition))

        spike_columns = [col for col in state_with_spikes.columns if '_spike_count' in col]
        
        # Define fixed time axis for all trials (based on expected 60Hz)
        expected_bins = int((PSTH_PRE + PSTH_POST) * 60)
        time_axis = np.linspace(-PSTH_PRE, PSTH_POST, expected_bins)
        time_col_names = [f't_{t:.3f}' for t in time_axis]

        n_collisions = 0
        # Process each neuron completely independently
        for col in spike_columns:
            area = get_simplified_area(col)

            # Track trials and their actual bin times for alignment
            valid_trials_data = []
            
            for event_idx, (trial_id, event, condition) in enumerate(valid_trial_rows):
                start = event - PSTH_PRE
                end = event + PSTH_POST
                
                # Get both spike counts AND bin times
                window_mask = (state_with_spikes['Bin'] >= start) & (state_with_spikes['Bin'] < end)
                window_spike_counts = state_with_spikes.loc[window_mask, col].values
                window_bin_times = state_with_spikes.loc[window_mask, 'Bin'].values
                
                # Check if we have enough bins
                min_acceptable_bins = int(expected_bins * MIN_BINS_RATIO)
                if len(window_spike_counts) < min_acceptable_bins:
                    continue

                firing_rates_aligned, n_bins_actual, collided = align_bins(
                    window_bin_times, window_spike_counts, event, PSTH_PRE, expected_bins)
                n_collisions += int(collided)

                valid_trials_data.append({
                    'event_idx': event_idx,
                    'trial_id': trial_id,
                    'event_time': event,
                    'condition': condition,
                    'firing_rates': firing_rates_aligned,
                    'n_bins': n_bins_actual
                })
            
            if len(valid_trials_data) < 10:  # Skip neuron if too few trials
                continue

            # Create one record per trial per neuron
            for trial_data in valid_trials_data:
                record = {
                    'pid': pid_name,
                    'session': session_name,
                    'neuron_id': col,
                    'area': area,
                    'trial_id': trial_data['trial_id'],  # PRESERVE ORIGINAL trial_id
                    'event_time': trial_data['event_time'],
                    'condition': trial_data['condition'],
                    'n_bins': trial_data['n_bins']
                }

                # Add firing rate for each time bin (with NaNs in right places)
                for col_name, fr in zip(time_col_names, trial_data['firing_rates']):
                    record[col_name] = fr

                summary_records.append(record)
        
        # Build DataFrame once after all neurons are processed
        session_data = pd.DataFrame(summary_records)
                            
        # Save per session
        with open(save_filename, 'wb') as f:
            pickle.dump(session_data, f)
        
        processed_count += 1
        print(f"Processed PID: {pid_name}")
        print(f"  Total valid trials: {len(valid_trial_rows)}/{len(trial_order)} "
              f"({100*len(valid_trial_rows)/max(len(trial_order),1):.1f}%)")
        if n_collisions:
            print(f"  Bin-position collisions averaged (not overwritten) in {n_collisions} neuron-trial windows")
        
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n" + "="*60)
print(f"PROCESSING SUMMARY")
print(f"="*60)
print(f"Newly processed: {processed_count}")
print(f"Already existed (skipped): {skipped_count}")
print(f"Total PIDs: {len(lda_pid)}")
if failed_sessions:
    print(f"Sessions with unfetchable trials (all their pids were skipped): {len(failed_sessions)}")


Processed PID: d14f70e6-bf7b-4d6d-a380-bfd0a46ed7a1
  Total valid trials: 400/400 (100.0%)


/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:436: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


Processed PID: 5d570bf6-a4c6-4bf1-a14b-2c878c84ef0e
  Total valid trials: 400/400 (100.0%)
Processed PID: f7c93877-ec05-4091-a003-e69fae0f2fa8
  Total valid trials: 400/400 (100.0%)


/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:436: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


Processed PID: 675952a4-e8b3-4e82-a179-cc970d5a8b01
  Total valid trials: 400/400 (100.0%)
Processed PID: 79f44ba1-c931-4346-82eb-f628a9374045
  Total valid trials: 400/400 (100.0%)

PROCESSING SUMMARY
Newly processed: 380
Already existed (skipped): 0
Total PIDs: 380
